# Plots of Suplementary Figure 2 and 3 and Extended data Figure 1 related to SNR.

In [ ]:
import pathlib
import re


import astropy.units as u
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from adjustText import adjust_text
from astropy.coordinates import Galactocentric, SkyCoord, get_sun
from astropy.time import Time
from matplotlib import rc
from matplotlib.lines import Line2D
from scipy.stats import gaussian_kde
from scipy.interpolate import interp1d

import utilities.plot_settings

Setting up plot properties:

In [ ]:
yellow = plt.cm.Set2(5 / 7)
orange = plt.cm.Set2(1 / 7)
sea = plt.cm.Set2(0 / 7)
blue = plt.cm.Set2(2 / 7)
green = plt.cm.Set2(4 / 7)

rc("text", usetex=True)
rc("font", family="serif")
plt.rcParams["text.latex.preamble"] = r"\usepackage{amsmath}"

SMALL_SIZE = 10
MEDIUM_SIZE = 20
BIGGER_SIZE = 28

plt.rc("font", size=SMALL_SIZE)  # controls default text sizes
plt.rc("axes", titlesize=BIGGER_SIZE)  # fontsize of the axes title
plt.rc("axes", labelsize=BIGGER_SIZE)  # fontsize of the x and y labels
plt.rc("xtick", labelsize=BIGGER_SIZE)  # fontsize of the tick labels
plt.rc("ytick", labelsize=BIGGER_SIZE)  # fontsize of the tick labels
plt.rc("legend", fontsize=MEDIUM_SIZE)  # legend fontsize
plt.rc("figure", titlesize=BIGGER_SIZE)  # fontsize of the figure title

# Reading data

In [ ]:
data = pd.read_csv('/Users/celsapardoaraujo/Documents/ML-Poppyns/data/paper_results/pardo_etal_2026/final_snr_psr_assoc_dataset.csv')

In [ ]:
# Read the full ATNF catalog.csv file. Binary pulsars are excluded.
df_atnf = pd.read_csv(
    "/Users/celsapardoaraujo/Documents/ML-Poppyns/data/observations/atnf_full_nobinary_25-03-2025_with_errors.csv",
    delimiter=";",
    header=[0, 1],
)
df_atnf.columns = df_atnf.columns.droplevel(1)

In [ ]:
# Select only stars that are not in globular clusters or in the Magellanic Clouds.
discard = [
    "EXGAL:SMC",
    "EXGAL:LMC",
    "GC:47Tuc",
    "GC:M3",
    "GC:M5",
    "GC:M13",
    "GC:NGC6440",
    "GC:Ter5",
    "GC:NGC6441",
    "GC:NGC6517",
    "GC:NGC6522",
    "GC:NGC6624",
    "GC:M28(NGC6626)",
    "GC:NGC6652",
    "GC:M22(NGC6656)",
    "GC:NGC6752",
    "GC:NGC6760",
    "GC:M15",
    "GC:M30",
]

df_atnf = df_atnf[
    ~df_atnf["ASSOC"].str.match("|".join(discard))
]

# Select only isolated non-recycled neutron stars through filters with P > 0.01 and Pdot > 1e-19 (for those with measured values).
df_atnf = df_atnf[df_atnf["P0"].to_numpy().astype(np.float64) > 0.01]

df_atnf = df_atnf[
    (df_atnf["P1"].to_numpy().astype(np.float64) > 1.0e-19)
    | (df_atnf["P1"].isin(["NAN"]))
]




RA_atnf = df_atnf["RAJD"].to_numpy().astype(np.float64)
DEC_atnf = df_atnf["DECJD"].to_numpy().astype(np.float64)
l_atnf = df_atnf["Gl"].to_numpy().astype(np.float64)
b_atnf = df_atnf["Gb"].to_numpy().astype(np.float64)
P_atnf = df_atnf["P0"].to_numpy().astype(np.float64)
Pdot_atnf = df_atnf["P1"].to_numpy().astype(np.float64)
DM_atnf = df_atnf["DM"].to_numpy().astype(np.float64)
dist_atnf = df_atnf["DIST"].to_numpy().astype(np.float64)
S1400_atnf = df_atnf["S1400"].to_numpy().astype(np.float64)

### Load a dynamical database wihtout kick velocity to mimic the SNR in our Galaxy

In [ ]:
output_dir = "../../data/paper_results/pardo_etal_2026/dyn_spiral_arm/"
data_dyn = pd.read_csv(
    pathlib.Path().joinpath(output_dir, "final_pop_dyn.csv"),
    delimiter=",",
    header=[0, 1],
)
data_dyn.columns

r = data_dyn["r"]["[kpc]"].to_numpy()
phi = data_dyn["phi"]["[rad]"].to_numpy()
z_sim = data_dyn["z"]["[kpc]"].to_numpy()

x_sim = r * np.cos(phi)
y_sim = r * np.sin(phi)

# Plot Suplementary Figure 3

In [ ]:
# Separating the snr with and without association and filter for those younger than 10 kyrs.
mask_assoc_ps = pd.notna(data['ns_name']) 
mask_age = data['age_max (yr)']<10000
data_assoc = data[mask_assoc_ps & mask_age]
data_no_assoc = data[~mask_assoc_ps & mask_age]

In [ ]:
# Extracting the x, y and z, with the error in the distance towards the sun.
def get_cartesian_with_error(df):
    dist_mean = (df['distance_min (kpc)'] + df['distance_max (kpc)']) / 2
    dist_min = df['distance_min (kpc)']
    dist_max = df['distance_max (kpc)']

    coords = SkyCoord(
        ra=df['J2000_ra (hh:mm:ss)'].values,
        dec=df['J2000_dec (dd:mm:ss)'].values,
        unit=(u.hourangle, u.deg),
        distance=dist_mean.values * u.kpc,
        frame='icrs'
    )
    coords_min = SkyCoord(
        ra=df['J2000_ra (hh:mm:ss)'].values,
        dec=df['J2000_dec (dd:mm:ss)'].values,
        unit=(u.hourangle, u.deg),
        distance=dist_min.values * u.kpc,
        frame='icrs'
    )
    coords_max = SkyCoord(
        ra=df['J2000_ra (hh:mm:ss)'].values,
        dec=df['J2000_dec (dd:mm:ss)'].values,
        unit=(u.hourangle, u.deg),
        distance=dist_max.values * u.kpc,
        frame='icrs'
    )

    galcen     = coords.transform_to(Galactocentric())
    galcen_min = coords_min.transform_to(Galactocentric())
    galcen_max = coords_max.transform_to(Galactocentric())

    return (
        galcen.x.value, galcen.y.value, galcen.z.value,
        galcen_min.x.value, galcen_min.y.value, galcen_min.z.value,
        galcen_max.x.value, galcen_max.y.value, galcen_max.z.value
    )

# Get coordinates for PSR and non-PSR data
(x_psr, y_psr, z_psr,
 x_psr_min, y_psr_min, z_psr_min,
 x_psr_max, y_psr_max, z_psr_max) = get_cartesian_with_error(data_assoc)

(x_nonpsr, y_nonpsr, z_nonpsr,
 x_nonpsr_min, y_nonpsr_min, z_nonpsr_min,
 x_nonpsr_max, y_nonpsr_max, z_nonpsr_max) = get_cartesian_with_error(data_no_assoc)

# Sun's Galactocentric position
sun = get_sun(Time('2025-05-14'))
sun_galcen = sun.transform_to(Galactocentric)
x_sun = sun_galcen.x.to('kpc').value
y_sun = sun_galcen.y.to('kpc').value
z_sun = sun_galcen.z.to('kpc').value

# Mean ages in 100-year units.
age_psr = ((data_assoc['age_max (yr)'] + data_assoc['age_min (yr)']) / 2) / 1000
age_nonpsr = ((data_no_assoc['age_max (yr)'] + data_no_assoc['age_min (yr)']) / 2) / 1000

# Normalize all ages together.
all_ages = np.concatenate([age_psr, age_nonpsr])
norm = mcolors.Normalize(vmin=all_ages.min(), vmax=all_ages.max())
cmap = cm.plasma

###--------------------------PLOT---------------------------------------
fig = plt.figure(figsize=(20, 10))
gs = gridspec.GridSpec(1, 4, width_ratios=[1, 1, 0.05, 0.03], wspace=0.6)

ax1 = fig.add_subplot(gs[0])

# --- Subplot 1: X–Y 
xy = np.vstack([y_sim, -x_sim])
kde_xy = gaussian_kde(xy)(xy)
ax1.scatter(y_sim, -x_sim, s=5, color='tab:blue', alpha=0.1, rasterized = True)


# Non-PSR
for x0, y0, x1, y1, xm, ym, age in zip(
    x_nonpsr_min, y_nonpsr_min, x_nonpsr_max, y_nonpsr_max, x_nonpsr, y_nonpsr, age_nonpsr
):
    color = cmap(norm(age))
    ax1.plot([x0, x1], [y0, y1], color=color)
    ax1.plot(xm, ym, 'o', color=color, markersize=5)

# PSR
for x0, y0, x1, y1, xm, ym, age in zip(
    x_psr_min, y_psr_min, x_psr_max, y_psr_max, x_psr, y_psr, age_psr
):
    color = cmap(norm(age))
    ax1.plot([x0, x1], [y0, y1], color=color)
    ax1.plot(xm, ym, '*', color=color, markersize=10)

# Sun and 2 kpc circle
circle1 = plt.Circle((x_sun, y_sun), 2, color='black', fill=False, linestyle='--', linewidth=1, label='2 kpc')
ax1.add_artist(circle1)
ax1.scatter(x_sun, y_sun, color='green', marker='D', s=100, label='Sun', zorder=10, rasterized = True)
ax1.set_aspect('equal')
ax1.set_xlabel('X [kpc]')
ax1.set_ylabel('Y [kpc]', labelpad=1)
ax1.grid(True)
ax1.set_xlim(-15, 15)
ax1.set_ylim(-15, 15)

# Custom legend for left plot
legend_elements = [
    Line2D([0], [0], marker='*', color='black', label='SNR with NS assoc ',
           markersize=12, linestyle='None'),
    Line2D([0], [0], marker='o', color='black', label='SNR no assoc ',
           markersize=8, linestyle='None'),
    Line2D([0], [0], marker='D', color='green', label='Sun ', markersize=10, linestyle='None'),
    Line2D([0], [0], color='black', lw=1, linestyle='--', label='2 kpc ')
]

# Add the legend to the upper right corner
ax1.legend(handles=legend_elements, loc='upper right', frameon=True)
ax2 = fig.add_subplot(gs[1])

# --- Right subplot: X-Z density ---
xz = np.vstack([y_sim, z_sim])
kde_xz = gaussian_kde(xz)(xz)
ax2.scatter(y_sim, z_sim, color='tab:blue', s=5, alpha=0.1, rasterized = True)
# Non-PSR
for x0, z0, x1, z1, xm, zm, age in zip(
    x_nonpsr_min, z_nonpsr_min, x_nonpsr_max, z_nonpsr_max, x_nonpsr, z_nonpsr, age_nonpsr
):
    color = cmap(norm(age))
    ax2.plot([x0, x1], [z0, z1], color=color)
    ax2.plot(xm, zm, 'o', color=color, markersize=5)

# PSR
for x0, z0, x1, z1, xm, zm, age in zip(
    x_psr_min, z_psr_min, x_psr_max, z_psr_max, x_psr, z_psr, age_psr
):
    color = cmap(norm(age))
    ax2.plot([x0, x1], [z0, z1], color=color)
    ax2.plot(xm, zm, '*', color=color, markersize=10)

# Sun and 2 kpc circle
circle2 = plt.Circle((x_sun, z_sun), 2, color='black', fill=False, linestyle='--', linewidth=1)
ax2.add_artist(circle2)
ax2.scatter(x_sun, z_sun, color='green', marker='D', s=100, label='Sun', zorder=10)

ax2.set_xlabel('X [kpc]')
ax2.set_ylabel('Z [kpc]', labelpad=2)
ax2.yaxis.set_label_position("left")
ax2.grid(True)
ax2.set_xlim(-15, 15)

# Center the right subplot vertically
pos1 = ax1.get_position()  # position of left subplot
pos2 = ax2.get_position()  # original position of right subplot

new_height = pos2.height * 0.5
new_y0 = pos1.y0 + (pos1.height - new_height) / 2
ax2.set_position([pos2.x0, new_y0, pos2.width, new_height])

# --- Colorbar: full left subplot height, close to right panel ---
cax_width = 0.02
cax_padding = 0.005
cax = fig.add_axes([pos2.x0 + pos2.width + cax_padding, pos1.y0, cax_width, pos1.height])
sm = cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = fig.colorbar(sm, cax=cax)
cbar.set_label('Mean age [kyr]')


fig.savefig(
    'plots/Suplementary_figure3.pdf',
    dpi=300,
    bbox_inches='tight'
)
plt.show()

Using the results of notebook: `compute_CCSN_birth_rate.ipynb`

Birthrate: median and 1 sigma =  (1.07, 2.0, 3.19)

Birthrate: median and 3 sigma  =  (0.94, 2.0, 7.29)

# Plot Extended data Figure 1

In [ ]:
def ecdf(data):
    sorted_ages = np.sort(data)
    cumulative_number_ages = np.arange(len(data))
    return sorted_ages, cumulative_number_ages


# Cumulative histogram of observed minimum and maximum ages
age_min_kyr = data_assoc['age_min (yr)'] / 1000
age_max_kyr = data_assoc['age_max (yr)'] / 1000

x_min, y_min = ecdf(age_min_kyr)
x_max, y_max = ecdf(age_max_kyr)

# Interpolate onto a common x-grid
x_common = np.linspace(0, 10, 500)

y_min_interp = interp1d(
    x_min,
    y_min,
    kind='previous',
    bounds_error=False,
    fill_value=(0, y_min[-1])
)(x_common)

y_max_interp = interp1d(
    x_max,
    y_max,
    kind='previous',
    bounds_error=False,
    fill_value=(0, y_max[-1])
)(x_common)

plt.figure(figsize=(8, 6))


# Overplot shaded area between curves
plt.fill_between(
    x_common,
    y_min_interp,
    y_max_interp,
    color=sea,
    alpha=0.4,
    label='Observed SNR age'
)

# Reference SN birthrate line
x_ref = np.arange(0, 11, 1)
y_ref_06 = x_ref * 10 * 0.6

plt.plot(
    x_ref,
    y_ref_06,
    color='tab:green',
    linestyle='--',
    lw=2
)

# Simulated birthrate with uncertainty bands
median = 2
x = np.arange(0, 11, 1)

y_mean = x * 10 * median
y_lower_1sigma = x * 10 * 1.07
y_upper_1sigma = x * 10 * 3.19
y_lower_3sigma = x * 10 * 0.94
y_upper_3sigma = x * 10 * 7.29

plt.plot(
    x,
    y_mean,
    color="#e6550d",
    linestyle='--',
    lw=2
)

plt.fill_between(
    x,
    y_lower_3sigma,
    y_upper_3sigma,
    color=orange,
    alpha=0.3,
    label=r'$3\sigma$ scaled volume BR'
)

plt.fill_between(
    x,
    y_lower_1sigma,
    y_upper_1sigma,
    color="#e6550d",
    alpha=0.3,
    label=r'$1\sigma$ scaled volume BR'
)

plt.text(
    0.8,
    24,
    'median scaled volume BR',
    color="#e6550d",
    fontsize=17,
    rotation=70
)

# Axis labels and formatting
plt.xlabel('SNR Age (kyr)')
plt.ylabel('Cumulative Number of SNR')
plt.xticks(np.arange(0, 11, 1))
plt.yticks(np.arange(0, 100, 10))
plt.ylim(0, 50)
plt.xlim(0, 10)
plt.grid(True)
plt.legend()
plt.tight_layout()

plt.savefig('plots/age_cum_assoc.pdf')
plt.show()


# Plot Supplementary Figure 2

In [ ]:
df = pd.read_csv( "../../data/paper_results/pardo_etal_2026/Table1.csv")

def parse_age_and_yerr(age_str):
    if age_str is None or (isinstance(age_str, float) and np.isnan(age_str)):
        return np.nan, np.nan, np.nan

    s = str(age_str).replace("−", "-").replace("–", "-")

    m = re.match(r"^([0-9]*\.?[0-9]+)\s*-\s*([0-9]*\.?[0-9]+)", s)
    if m:
        a = float(m.group(1))
        b = float(m.group(2))
        y = 0.5 * (a + b)
        return y, y - a, b - y

    m = re.match(r"^([0-9]*\.?[0-9]+)", s)
    if m:
        y = float(m.group(1))
        return y, max(y - 0, 0), max(9 - y, 0)

    return np.nan, np.nan, np.nan

parsed = df["snr_age_str"].apply(parse_age_and_yerr)
df["snr_age_kyr"] = parsed.apply(lambda t: t[0])
df["snr_err_low"] = parsed.apply(lambda t: t[1])
df["snr_err_high"] = parsed.apply(lambda t: t[2])

plot_df = df.dropna(subset=["snr_age_kyr", "tau_c_kyr", "B14"]).copy()
plot_df["Bdip"] = plot_df["B14"] * 1e14

# ----------------------------
# 3) Plot setup
# ----------------------------
markers = {"Mag": "o", "RPP": "s"}
fig, ax = plt.subplots(figsize=(9, 6.5))
sc_for_cbar = None

norm = mcolors.LogNorm(
    vmin=plot_df["Bdip"].min(),
    vmax=plot_df["Bdip"].max()
)


# Scatter + error bars
for t, g in plot_df.groupby("type"):
    x = g["snr_age_kyr"].to_numpy()
    y = g["tau_c_kyr"].to_numpy()
    xerr = np.vstack([g["snr_err_low"], g["snr_err_high"]])

    sc_for_cbar = ax.scatter(
        x, y,
        c=g["Bdip"],
        cmap="viridis",
        norm=norm,
        marker=markers[t],
        s=90,
        edgecolors="k",
        linewidths=0.6,
        label=t,
        zorder=3
    )

    ax.errorbar(
        x, y,
        xerr=xerr,
        fmt="none",
        ecolor="0.6",
        elinewidth=1,
        capsize=2,
        zorder=2
    )

# x = y line
xmin = min(plot_df["snr_age_kyr"].min(), plot_df["tau_c_kyr"].min())
xmax = max(plot_df["snr_age_kyr"].max(), plot_df["tau_c_kyr"].max())

xline = np.logspace(np.log10(xmin), np.log10(xmax), 300)
ax.plot(xline, xline, "k--", linewidth=1.4)

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("SNR age [kyr]")
ax.set_ylabel(r"Characteristic age $\tau_{\rm c}$ [kyr]")
ax.grid(True, which="both", alpha=0.3)

xlow, xhigh = 0.5, 2e1   # x-axis is now SNR age
ylow, yhigh = 0.5, 4e1   # y-axis is now tau_c

# shaded region: either SNR age < 2 kyr or tau_c < 2 kyr
ax.axvspan(xlow, 2.0, alpha=0.05, zorder=0)
ax.axhspan(ylow, 2.0, alpha=0.05, zorder=0)

ax.set_xlim(xlow, xhigh)
ax.set_ylim(ylow, yhigh)

# Labels ----------------------------
manual_offsets = {
    "PSR B1509-58":   dict(xytext=(0, 8),   ha="center", va="bottom"),
    "PSR J1747-2809": dict(xytext=(8, 0),   ha="left",   va="center"),
    "PSR J1400-6325": dict(xytext=(0, -8),  ha="center", va="top"),
    "PSR J1833-1034": dict(xytext=(20, 0), ha="left", va="center"),
    "1E 1841-045":    dict(xytext=(0, -8),   ha="center",   va="top"),
    "PSR J1811-1925": dict(xytext=(0, 8),   ha="center", va="bottom"),
    "PSR J0205+6449": dict(xytext=(0, 8),   ha="center", va="bottom")
}

for _, row in plot_df.iterrows():
    xi = row["snr_age_kyr"]
    yi = row["tau_c_kyr"]
    name = row["name"]

    if name in manual_offsets:
        opts = manual_offsets[name]
        ax.annotate(
            name,
            xy=(xi, yi),
            xytext=opts["xytext"],
            textcoords="offset points",
            ha=opts["ha"],
            va=opts["va"],
            fontsize=8,
            zorder=4
        )
    else:
        ax.annotate(
            name,
            xy=(xi, yi),
            xytext=(5, 5),
            textcoords="offset points",
            ha="left",
            va="bottom",
            fontsize=8,
            zorder=4
        )

# Colorbar ----------------------------
cbar = fig.colorbar(sc_for_cbar, ax=ax, pad=0.02)
cbar.set_label(r"$B_{\rm dip}\ [\mathrm{G}]$")

# Legend ----------------------------
legend_handles = [
    Line2D([0], [0],
           marker='o',
           color='none',
           markerfacecolor='k',
           markeredgecolor='k',
           markersize=8,
           label='Magnetar'),
    Line2D([0], [0],
           marker='s',
           color='none',
           markerfacecolor='k',
           markeredgecolor='k',
           markersize=8,
           label='RPP')
]

ax.legend(handles=legend_handles, frameon=True, loc='upper right')

plt.tight_layout()
plt.savefig('plots/real_vs_charact_Bdip_2kyrs_flipped.pdf')
plt.show()